# Mini-Project MP03 — Press Release to Plot

## Industry Comparison: Financial Services and Travel and Hospitality

*CIS 3120 — Programming for Analytics*
*Baruch College, Zicklin School of Business*

---

**Team number:** 16

**Team members:**
- Financial Services Pipeline Lead: Chinmoy Chowdhury
- Travel and Hospitality Pipeline Lead: Chinmoy/Daniel 
- Comparison and Visualization Lead (Integrator): Daniel Wang

**Submission filename:** MP03_Notebook_team_16.ipynb

---

## How to use this starter

1. Make a copy of this notebook and rename it MP03_Notebook_team_16.ipynb.
2. Confirm the User-Agent email in the setup cell is correct for SEC EDGAR and Nominatim requests.
3. Configure your Anthropic API key in Colab Secrets as ANTHROPIC_API_KEY.
4. Work through the notebook section by section. Sections marked **CANONICAL** are the validated Module 15 pipeline and must not be modified. Sections marked **TODO** are where your team writes new code.
5. Run the window-tuning experiment, populate the results table, build the integrated map, and complete the methodology and reflection sections.
6. Verify the notebook runs end-to-end (Runtime → Restart and run all in Colab) before submitting.

See docs/MP03_Assignment.docx for the full assignment specification.

---

## 1. Setup

Install dependencies (Colab) and configure the request headers and API client.

In [1]:
# Colab installs (silent). The other packages are pre-installed in the Colab base image.
!pip install anthropic folium beautifulsoup4 --quiet


In [2]:
import json
import re
import os
import time
from datetime import date, datetime, timedelta

import requests
from bs4 import BeautifulSoup
import folium
import pandas as pd
from anthropic import Anthropic
from IPython.display import display

# ─────────────────────────────────────────────────────────────────────────
# CRITICAL: Keep the email below current and descriptive.
# Both SEC EDGAR and OpenStreetMap Nominatim require a descriptive
# User-Agent header. Generic agents are rejected with HTTP 403.
# ─────────────────────────────────────────────────────────────────────────
USER_AGENT = os.environ.get("SEC_USER_AGENT", "CIS3120 MP03 Team 16 - and.re.i.asa.nches.x@gmail.com")

REQUEST_HEADERS = {"User-Agent": USER_AGENT}

# ─────────────────────────────────────────────────────────────────────────
# Endpoints and constants
# ─────────────────────────────────────────────────────────────────────────
EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"
NOMINATIM_URL    = "https://nominatim.openstreetmap.org/search"

EDGAR_PAUSE      = 0.15   # seconds between EDGAR requests (SEC: 10 req/sec)
NOMINATIM_PAUSE  = 1.10   # seconds between Nominatim requests (1 req/sec)

# Anthropic model: current Haiku in the Claude 4.5 family.
MODEL_ID = "claude-haiku-4-5-20251001"


In [3]:
# Configure the Anthropic API client. Works in Colab Secrets or local .env.local.
def load_env_value(name, paths=("../.env.local", ".env.local")):
    value = os.environ.get(name)
    if value:
        return value.strip().strip("'").strip('"')

    for env_path in paths:
        if not os.path.exists(env_path):
            continue
        with open(env_path) as env_file:
            for line in env_file:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                if line.startswith("export "):
                    line = line[len("export "):].strip()
                if not line.startswith(name):
                    continue
                value = line[len(name):].strip()
                if value.startswith("=") or value.startswith(":"):
                    value = value[1:].strip()
                if value:
                    return value.strip().strip("'").strip('"')
    return None

try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
except ModuleNotFoundError:
    ANTHROPIC_API_KEY = load_env_value("ANTHROPIC_API_KEY")

if not ANTHROPIC_API_KEY:
    raise ValueError("ANTHROPIC_API_KEY is missing. Add it to Colab Secrets or .env.local.")

client = Anthropic(api_key=ANTHROPIC_API_KEY)


In [4]:
# ─────────────────────────────────────────────────────────────────────────
# Industry ticker lists and search-phrase lists (seeded defaults + extensions)
# Defined directly here so there is no dependency on an external mp03 module.
# ─────────────────────────────────────────────────────────────────────────

FINANCIAL_SERVICES_TICKERS = [
    # Money-center banks
    "JPM", "BAC", "WFC", "C",
    # Regional banks
    "PNC", "USB", "TFC",
    # Asset management
    "BLK", "BX",
    # Insurance
    "MET", "PRU",
    # Payments
    "V", "MA", "AXP",
    # Capital markets (seed undercounted these)
    "GS", "MS", "SCHW",
    # Regional/community banks seen in branch-opening filings
    "NKSH", "UNTY", "FMCB", "HNVR", "MCB", "FUNC",
    "WSBK", "WSBC", "PGC", "JUVF",
    # Insurance brokerage
    "BRO",
]

FINANCIAL_SERVICES_PHRASES = [
    '"new branch"',
    '"branch opening"',
    '"branch closure"',
    '"branch closing"',
    '"branch consolidation"',
    '"regional office"',
    '"office closure"',
    '"operations center"',
    '"data center"',
    '"new location"',
    '"financial center"',
    '"wealth management office"',
]

TRAVEL_HOSPITALITY_TICKERS = [
    # Hotels
    "MAR", "HLT", "H", "CHH", "WH",
    # Cruise
    "CCL", "RCL", "NCLH",
    # Airlines
    "DAL", "UAL", "AAL", "LUV",
    # Online travel
    "BKNG", "EXPE",
    # Gaming, venue, lodging, and hospitality real estate operators
    "PENN", "BALY", "GLPI", "VENU", "CWD", "HHH", "TH",
]

TRAVEL_HOSPITALITY_PHRASES = [
    '"new property"',
    '"new hotel"',
    '"hotel opening"',
    '"resort opening"',
    '"property opening"',
    '"brand conversion"',
    '"new route"',
    '"new gateway"',
    '"new terminal"',
    '"grand opening"',
]

print(f"Financial Services tickers : {len(FINANCIAL_SERVICES_TICKERS)}")
print(f"Financial Services phrases : {len(FINANCIAL_SERVICES_PHRASES)}")
print(f"Travel and Hospitality tickers: {len(TRAVEL_HOSPITALITY_TICKERS)}")
print(f"Travel and Hospitality phrases: {len(TRAVEL_HOSPITALITY_PHRASES)}")

Financial Services tickers : 28
Financial Services phrases : 12
Travel and Hospitality tickers: 21
Travel and Hospitality phrases: 10


---

## 2. Canonical Pipeline (Module 15)

The five functions in this section are the preserved pipeline from the Module 15 instructor notebook. **Do not modify these signatures.** Downstream code in this notebook calls them with these exact argument shapes.

### Stage 1 — Retrieve candidate 8-K filings from EDGAR

Each phrase is queried independently. Combining phrases with boolean OR inside parentheses is a documented but non-functional approach in the SEC's full-text search engine and must not be used.

In [5]:
def search_edgar_one_phrase(
    phrase: str,
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
) -> tuple[list[dict], int]:
    """Query EDGAR full-text search for one phrase across a date window.

    Returns a tuple of (list of hit dicts, total reported by EDGAR).
    """
    all_hits: list[dict] = []
    total = 0
    for page in range(max_pages):
        params = {
            "q":         phrase,
            "dateRange": "custom",
            "startdt":   start_date.isoformat(),
            "enddt":     end_date.isoformat(),
            "forms":     forms,
            "from":      page * 100,
        }
        response = requests.get(
            EDGAR_SEARCH_URL,
            params=params,
            headers=REQUEST_HEADERS,
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        hits = data.get("hits", {}).get("hits", [])
        all_hits.extend(hits)
        total = data.get("hits", {}).get("total", {}).get("value", 0)
        if (page + 1) * 100 >= total:
            break
        time.sleep(EDGAR_PAUSE)
    return all_hits, total

In [6]:
def search_edgar_all_phrases(
    phrases: list[str],
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
    max_filings: int = 250,
) -> list[dict]:
    """Run search_edgar_one_phrase across a list of phrases with retry-with-backoff.

    Deduplicates by (accession number, exhibit filename). Stops accumulating
    once max_filings is reached.
    """
    seen: set[str] = set()
    deduped: list[dict] = []
    backoff_waits = [5, 10, 15]

    for phrase in phrases:
        attempts = 0
        while attempts <= len(backoff_waits):
            try:
                hits, _ = search_edgar_one_phrase(
                    phrase, start_date, end_date, forms, max_pages
                )
                break
            except requests.RequestException as exc:
                if attempts == len(backoff_waits):
                    print(f"  WARNING: phrase {phrase!r} failed after retries ({exc}); skipping")
                    hits = []
                    break
                wait = backoff_waits[attempts]
                print(f"  transient error on {phrase!r}: {exc}. retrying in {wait}s...")
                time.sleep(wait)
                attempts += 1

        for hit in hits:
            key = hit.get("_id", "")
            if key and key not in seen:
                seen.add(key)
                deduped.append(hit)
            if len(deduped) >= max_filings:
                return deduped
        time.sleep(EDGAR_PAUSE)

    return deduped

### Stage 2 — Fetch the press release text from each filing

In [7]:
def build_exhibit_url(hit: dict) -> str:
    """Construct the SEC archive URL for the exhibit referenced by the hit."""
    accession_full, filename = hit["_id"].split(":")
    accession_no_dashes = accession_full.replace("-", "")
    cik = hit["_source"]["ciks"][0].lstrip("0")
    return (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik}/{accession_no_dashes}/{filename}"
    )


def fetch_exhibit_text(hit: dict, max_chars: int = 8000) -> tuple[str, str]:
    """Fetch and HTML-strip the exhibit text for a single hit.

    Returns (text, url). Truncates at max_chars (~2000 tokens).
    """
    url = build_exhibit_url(hit)
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    if len(text) > max_chars:
        text = text[:max_chars] + " [\u2026truncated\u2026]"
    return text, url

### Stage 3 — Classify and extract with the Anthropic API

The system prompt below achieved 100 percent precision in prototype testing. Use it verbatim.

In [8]:
EXTRACTION_SYSTEM_PROMPT = """You are an analyst reviewing 8-K filing exhibits to identify announcements of location-related corporate events: openings, closings, relocations, or expansions of physical facilities (stores, warehouses, distribution centers, offices, plants).

Return ONLY a JSON object with these exact fields:
- is_location_event: boolean. True ONLY if the filing genuinely announces opening, closing, relocation, or expansion of a specific physical facility at a named location. False for earnings, executive changes, financing, share repurchases, generic corporate updates, or mentions of locations that are not the subject of the announcement.
- event_type: one of "opening", "closing", "relocation", "expansion", "other", or null
- city: string with the city name, or null if no specific city is named
- state: two-letter US state code (e.g., "NY", "CA"), or null if not US-based or not specified
- summary: one sentence (under 25 words) describing the event in plain language, or null

Be strict. If the filing mentions a location only in passing (e.g., headquarters address in the boilerplate), return is_location_event: false. Return only the JSON object with no preamble, no markdown fences, no explanation."""


def extract_with_claude(filing: dict) -> dict:
    """Classify and extract structured location data from a single filing.

    Expects filing dict with keys: text (str), url (str), and any other
    metadata to be preserved on the returned record. Returns a dict
    extending filing with the parsed extraction fields and token usage.
    """
    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=300,
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": filing["text"]}],
    )

    raw = response.content[0].text.strip()
    raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"is_location_event": False, "_parse_error": raw[:200]}

    record = {**filing, **parsed}
    record["input_tokens"]  = response.usage.input_tokens
    record["output_tokens"] = response.usage.output_tokens
    return record

### Stage 4 — Geocode the locations

Nominatim enforces a strict 1-request-per-second policy. The 1.10-second pause is a comfortable margin.

In [9]:
def geocode_location(city: str, state: str | None) -> tuple[float, float] | None:
    """Geocode a US city/state pair via OpenStreetMap Nominatim.

    Returns (latitude, longitude) on success, None if no match is found.
    """
    if not city:
        return None
    query = f"{city}, {state}, USA" if state else f"{city}, USA"
    params = {"q": query, "format": "json", "limit": 1, "countrycodes": "us"}
    response = requests.get(
        NOMINATIM_URL,
        params=params,
        headers=REQUEST_HEADERS,
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    time.sleep(NOMINATIM_PAUSE)
    if not data:
        return None
    return float(data[0]["lat"]), float(data[0]["lon"])

### Stage 5 — Render the folium map (base configuration)

The base map and event color palette are provided. Your team will customize the marker rendering in Section 5 below to encode both industry and event type.

In [10]:
EVENT_COLORS = {
    "opening":    "green",
    "closing":    "red",
    "relocation": "orange",
    "expansion":  "blue",
    "other":      "gray",
}

# Reasonable default center (geographic center of the contiguous US).
US_CENTER_LAT = 39.8
US_CENTER_LON = -98.6

---

## 3. Required New Functions

Each team adds the three functions below. Each one has a single, well-defined responsibility.

Reference: `docs/MP03_Assignment.docx`, Section 3.

In [11]:
def extract_hit_tickers(hit: dict) -> set[str]:
    """Return ticker symbols visible in an EDGAR search hit."""
    src = hit.get("_source", {})
    tickers: set[str] = set()

    for ticker in src.get("tickers", []) or []:
        if isinstance(ticker, str) and ticker.strip():
            tickers.add(ticker.strip().upper())

    for display_name in src.get("display_names", []) or []:
        if not isinstance(display_name, str):
            continue
        for group in re.findall(r"\(([^()]*)\)", display_name):
            if "CIK" in group.upper():
                continue
            for part in group.split(","):
                candidate = part.strip().upper()
                if re.fullmatch(r"[A-Z][A-Z0-9.-]{0,9}", candidate):
                    tickers.add(candidate)

    return tickers


def primary_ticker_for_hit(hit: dict, ticker_list: list[str]) -> str | None:
    """Choose the ticker from a hit that belongs to the requested industry list."""
    allowed = {ticker.upper() for ticker in ticker_list}
    hit_tickers = extract_hit_tickers(hit)

    matched = hit.get("_matched_ticker")
    if isinstance(matched, str) and matched.upper() in allowed:
        return matched.upper()

    for ticker in ticker_list:
        if ticker.upper() in hit_tickers:
            return ticker.upper()

    return sorted(hit_tickers)[0] if hit_tickers else None


def filter_candidates_by_tickers(
    candidates: list[dict],
    ticker_list: list[str],
) -> list[dict]:
    """Restrict a candidate set returned by Stage 1 to a list of tickers.

    EDGAR search hits are not consistent: some expose tickers in
    hit["_source"]["tickers"], while others only include them inside
    display_names such as "Company Name (JPM) (CIK ...)". This function
    checks both places and returns only hits whose tickers intersect
    ticker_list.

    Parameters
    ----------
    candidates : list[dict]
        EDGAR hits as returned by search_edgar_all_phrases.
    ticker_list : list[str]
        Tickers to retain (e.g., FINANCIAL_SERVICES_TICKERS).

    Returns
    -------
    list[dict]
        The subset of candidates whose tickers intersect ticker_list.
    """
    ticker_set = {ticker.upper() for ticker in ticker_list}
    filtered = []

    for hit in candidates:
        hit_tickers = extract_hit_tickers(hit)
        matches = hit_tickers & ticker_set
        if matches:
            hit_copy = {**hit, "_matched_ticker": sorted(matches)[0]}
            filtered.append(hit_copy)

    return filtered


In [12]:
def run_industry_pipeline(
    industry_label: str,
    ticker_list: list[str],
    phrase_list: list[str],
    window_days: int,
) -> list[dict]:
    """Run all five pipeline stages for one industry slice."""
    window_end = date.today()
    window_start = window_end - timedelta(days=window_days)

    print(f"[{industry_label}] Searching EDGAR: {window_start} to {window_end}", flush=True)

    all_hits_by_id = {}
    raw_hit_count = 0
    skipped_nonmatching = 0
    for ticker_index, ticker in enumerate(ticker_list, start=1):
        print(f"  searching {ticker} ({ticker_index}/{len(ticker_list)})", flush=True)
        ticker_upper = ticker.upper()
        for phrase in phrase_list:
            try:
                params = {
                    "q": phrase,
                    "dateRange": "custom",
                    "startdt": window_start.isoformat(),
                    "enddt": window_end.isoformat(),
                    "forms": "8-K",
                    "entity": ticker,
                    "from": 0,
                }
                response = requests.get(
                    EDGAR_SEARCH_URL,
                    params=params,
                    headers=REQUEST_HEADERS,
                    timeout=30,
                )
                response.raise_for_status()
                data = response.json()
                hits = data.get("hits", {}).get("hits", [])
                raw_hit_count += len(hits)
                for hit in hits:
                    if ticker_upper not in extract_hit_tickers(hit):
                        skipped_nonmatching += 1
                        continue
                    hit_copy = {**hit, "_matched_ticker": ticker_upper}
                    all_hits_by_id[hit["_id"]] = hit_copy
                time.sleep(EDGAR_PAUSE)
            except Exception as exc:
                print(f"  [warn] {ticker} / {phrase}: {exc}")

    candidates = list(all_hits_by_id.values())
    print(
        f"[{industry_label}] Candidates found: {len(candidates)} "
        f"({skipped_nonmatching} non-matching hits skipped)",
        flush=True,
    )

    events = []
    total_input_tokens = 0
    total_output_tokens = 0

    for i, hit in enumerate(candidates):
        try:
            text, url = fetch_exhibit_text(hit)
            src = hit["_source"]
            filing = {
                "company": (src.get("display_names") or ["(unknown)"])[0],
                "ticker": primary_ticker_for_hit(hit, ticker_list),
                "file_date": src.get("file_date"),
                "accession": hit["_id"].split(":")[0],
                "url": url,
                "text": text,
            }
            record = extract_with_claude(filing)
            total_input_tokens += record.get("input_tokens", 0)
            total_output_tokens += record.get("output_tokens", 0)

            if record.get("is_location_event"):
                events.append(record)

        except Exception as exc:
            print(f"  [warn] Failed on hit {i}: {exc}")

        time.sleep(EDGAR_PAUSE)

    estimated_cost = (
        total_input_tokens / 1_000_000 * 1.0
        + total_output_tokens / 1_000_000 * 5.0
    )

    print(f"[{industry_label}] Location events found: {len(events)}", flush=True)
    print(f"[{industry_label}] Estimated Stage 3 cost: ${estimated_cost:.4f}", flush=True)

    geocoded = []
    for event in events:
        coords = geocode_location(event.get("city"), event.get("state"))
        if coords:
            event["latitude"] = coords[0]
            event["longitude"] = coords[1]
            event["lat"] = coords[0]
            event["lon"] = coords[1]
            event["industry"] = industry_label
            geocoded.append(event)

    print(f"[{industry_label}] Geocoded: {len(geocoded)} of {len(events)}", flush=True)

    trial = {
        "candidate_count": len(candidates),
        "raw_hit_count": raw_hit_count,
        "skipped_nonmatching_count": skipped_nonmatching,
        "event_count": len(events),
        "geocoded_count": len(geocoded),
        "estimated_cost_usd": estimated_cost,
        "window_days": window_days,
    }
    run_industry_pipeline.last_trial = trial

    for event in geocoded:
        event["_trial_candidate_count"] = trial["candidate_count"]
        event["_trial_estimated_cost"] = trial["estimated_cost_usd"]
        event["_trial_event_count"] = trial["event_count"]

    return geocoded


In [13]:
def summarize_window_trial(
    industry_label: str,
    window_days: int,
    candidate_count: int,
    event_count: int,
    estimated_cost_usd: float,
) -> dict:
    """Record the result of one window-tuning trial.

    Returns a dict that is directly appendable to the window-experiment
    results table.

    Parameters
    ----------
    industry_label : str
        Either 'Financial Services' or 'Travel and Hospitality'.
    window_days : int
        One of 30, 60, 90, 180, 360.
    candidate_count : int
        Length of filtered candidate list before Stage 3.
    event_count : int
        Number of records where is_location_event is True.
    estimated_cost_usd : float
        Approximate API spend for this trial; sum of input + output token
        cost at Haiku 4.5 pricing ($1/M input, $5/M output).

    Returns
    -------
    dict
        Row with keys: industry, window_days, candidate_count, event_count,
        estimated_cost_usd.
    """
    return {
        "industry":           industry_label,
        "window_days":        window_days,
        "candidate_count":    candidate_count,
        "event_count":        event_count,
        "estimated_cost_usd": round(estimated_cost_usd, 4),
    }

---

## 4. Window-Tuning Experiment

Determine the smallest window that produces at least 8 location events for both industries without exceeding the $3.00 cumulative cost ceiling.

**Protocol:**
1. Begin at `WINDOW_DAYS = 30`. Run the pipeline for both industries.
2. If both industries reach the event-count target, stop.
3. Otherwise advance through 60, 90, 180, 360. Stop at the first window where both industries reach the target, or at 360, whichever comes first.

**Stopping criteria:**

| Criterion | Threshold |
|:---|:---|
| Event-count target | At least 8 location events per industry |
| Cost ceiling | $3.00 cumulative across all trials |
| Window ceiling | 360 days |

Reference: `docs/MP03_Assignment.docx`, Section 4.

In [14]:
# Initialize the window-experiment results table.
# Append one row per (industry, window) trial that you actually run.
window_results = pd.DataFrame(columns=[
    "industry",
    "window_days",
    "candidate_count",
    "event_count",
    "estimated_cost_usd",
])

display(window_results)


,industry,window_days,candidate_count,event_count,estimated_cost_usd


### 4.1 Window trials — Financial Services

Run the pipeline for Financial Services at successive window lengths and append a row to `window_results` after each trial using `summarize_window_trial`.

In [15]:
WINDOW_SIZES = [30, 60, 90, 180, 360]
EVENT_COUNT_TARGET = 8
COST_CEILING = 3.00

cumulative_cost = 0.0
fs_events = []
fs_window_days = None

for window in WINDOW_SIZES:
    print(f"\n=== Financial Services | window = {window} days ===")

    events = run_industry_pipeline(
        "Financial Services",
        FINANCIAL_SERVICES_TICKERS,
        FINANCIAL_SERVICES_PHRASES,
        window_days=window,
    )
    trial = run_industry_pipeline.last_trial

    row = summarize_window_trial(
        industry_label="Financial Services",
        window_days=window,
        candidate_count=trial["candidate_count"],
        event_count=trial["event_count"],
        estimated_cost_usd=trial["estimated_cost_usd"],
    )
    row_df = pd.DataFrame([row])
    if window_results.empty:
        window_results = row_df
    else:
        window_results = pd.concat([window_results, row_df], ignore_index=True)
    cumulative_cost += trial["estimated_cost_usd"]

    print(f"  Classified events: {trial['event_count']} | "
          f"Geocoded events: {trial['geocoded_count']} | "
          f"Trial cost: ${trial['estimated_cost_usd']:.4f} | "
          f"Cumulative: ${cumulative_cost:.4f}")

    fs_events = events
    fs_window_days = window

    if trial["event_count"] >= EVENT_COUNT_TARGET:
        print(f"  Target reached at {window} days. Stopping Financial Services trials.")
        break

    if cumulative_cost >= COST_CEILING:
        print("  Cost ceiling reached. Stopping Financial Services trials.")
        break

print(f"\nFinal Financial Services geocoded event count: {len(fs_events)}")
display(window_results)



=== Financial Services | window = 30 days ===
[Financial Services] Searching EDGAR: 2026-04-16 to 2026-05-16


  searching JPM (1/28)


  searching BAC (2/28)


  searching WFC (3/28)


  searching C (4/28)


...

[Financial Services] Location events found: 11


[Financial Services] Estimated Stage 3 cost: $0.0320


[Financial Services] Geocoded: 10 of 11


  Classified events: 11 | Geocoded events: 10 | Trial cost: $0.0320 | Cumulative: $0.0320
  Target reached at 30 days. Stopping Financial Services trials.

Final Financial Services geocoded event count: 10


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,12,11,0.032


### 4.2 Window trials — Travel and Hospitality

In [16]:
WINDOW_SIZES = [30, 60, 90, 180, 360]
EVENT_COUNT_TARGET = 8
COST_CEILING = 3.00

cumulative_cost = window_results["estimated_cost_usd"].astype(float).sum()
th_events = []
th_window_days = None

for window in WINDOW_SIZES:
    print(f"\n=== Travel and Hospitality | window = {window} days ===")

    events = run_industry_pipeline(
        "Travel and Hospitality",
        TRAVEL_HOSPITALITY_TICKERS,
        TRAVEL_HOSPITALITY_PHRASES,
        window_days=window,
    )
    trial = run_industry_pipeline.last_trial

    row = summarize_window_trial(
        industry_label="Travel and Hospitality",
        window_days=window,
        candidate_count=trial["candidate_count"],
        event_count=trial["event_count"],
        estimated_cost_usd=trial["estimated_cost_usd"],
    )
    row_df = pd.DataFrame([row])
    if window_results.empty:
        window_results = row_df
    else:
        window_results = pd.concat([window_results, row_df], ignore_index=True)
    cumulative_cost += trial["estimated_cost_usd"]

    print(f"  Classified events: {trial['event_count']} | "
          f"Geocoded events: {trial['geocoded_count']} | "
          f"Trial cost: ${trial['estimated_cost_usd']:.4f} | "
          f"Cumulative: ${cumulative_cost:.4f}")

    th_events = events
    th_window_days = window

    if trial["event_count"] >= EVENT_COUNT_TARGET:
        print(f"  Target reached at {window} days. Stopping Travel and Hospitality trials.")
        break

    if cumulative_cost >= COST_CEILING:
        print("  Cost ceiling reached. Stopping Travel and Hospitality trials.")
        break

print(f"\nFinal Travel and Hospitality geocoded event count: {len(th_events)}")
display(window_results)



=== Travel and Hospitality | window = 30 days ===
[Travel and Hospitality] Searching EDGAR: 2026-04-16 to 2026-05-16


  searching MAR (1/21)


  searching HLT (2/21)


  searching H (3/21)


  searching CHH (4/21)


...

[Travel and Hospitality] Location events found: 9


[Travel and Hospitality] Estimated Stage 3 cost: $0.0264


[Travel and Hospitality] Geocoded: 9 of 9


  Classified events: 9 | Geocoded events: 9 | Trial cost: $0.0264 | Cumulative: $0.0899
  Target reached at 180 days. Stopping Travel and Hospitality trials.

Final Travel and Hospitality geocoded event count: 9


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,12,11,0.0320
1,Travel and Hospitality,30,2,2,0.0052
2,Travel and Hospitality,60,3,3,0.0077
3,Travel and Hospitality,90,7,7,0.0187
4,Travel and Hospitality,180,10,9,0.0264


### 4.3 Selected window and final pipeline runs

Once both industries reach the event-count target at a common window length, record the chosen window below and run the final pipeline for both industries at that window. The events from these two final runs feed Section 5.

In [17]:
if not fs_events or not th_events:
    raise ValueError("Run both industry window-trial cells before this final step.")

CHOSEN_WINDOW_DAYS = max(fs_window_days, th_window_days)
print(f"Chosen common window: {CHOSEN_WINDOW_DAYS} days")

# If one industry hit the target earlier, rerun it at the common window.
if fs_window_days != CHOSEN_WINDOW_DAYS:
    fs_events = run_industry_pipeline(
        "Financial Services",
        FINANCIAL_SERVICES_TICKERS,
        FINANCIAL_SERVICES_PHRASES,
        window_days=CHOSEN_WINDOW_DAYS,
    )
    fs_window_days = CHOSEN_WINDOW_DAYS

if th_window_days != CHOSEN_WINDOW_DAYS:
    th_events = run_industry_pipeline(
        "Travel and Hospitality",
        TRAVEL_HOSPITALITY_TICKERS,
        TRAVEL_HOSPITALITY_PHRASES,
        window_days=CHOSEN_WINDOW_DAYS,
    )
    th_window_days = CHOSEN_WINDOW_DAYS

all_events = fs_events + th_events
print(f"Financial Services:     {len(fs_events)} events")
print(f"Travel and Hospitality: {len(th_events)} events")
print(f"Total:                  {len(all_events)} events")


Chosen common window: 180 days
[Financial Services] Searching EDGAR: 2025-11-17 to 2026-05-16


  searching JPM (1/28)


  searching BAC (2/28)


  searching WFC (3/28)


  searching C (4/28)


...

[Financial Services] Candidates found: 25 (14423 non-matching hits skipped)


[Financial Services] Location events found: 18


[Financial Services] Estimated Stage 3 cost: $0.0693


[Financial Services] Geocoded: 15 of 18


Financial Services:     15 events
Travel and Hospitality: 9 events
Total:                  24 events


---

## 5. Integrated Folium Map

Build a single map containing markers from both industries. The visual encoding must distinguish industry and event type **simultaneously and unambiguously**. The recommended scheme is:

- **Industry** by marker color family (e.g., navy for Financial Services, teal for Travel and Hospitality).
- **Event type** by marker icon shape (e.g., `home` for opening, `times-circle` for closing).

Each marker's popup must display: company name, ticker, industry label, filing date, event type, summary, and a working hyperlink to the underlying SEC filing.

Reference: `docs/MP03_Assignment.docx`, Section 7 (verification checklist).

In [18]:
if not all_events:
    raise ValueError("No events to map. Run the final pipeline cell first.")

INDUSTRY_COLORS = {
    "Financial Services": "darkblue",
    "Travel and Hospitality": "green",
}

EVENT_ICONS = {
    "opening": "plus-circle",
    "closing": "times-circle",
    "relocation": "exchange",
    "expansion": "expand",
    "other": "info-circle",
}

m = folium.Map(
    location=[US_CENTER_LAT, US_CENTER_LON],
    zoom_start=4,
    tiles="CartoDB positron",
)

marker_count = 0
for event in all_events:
    latitude = event.get("latitude", event.get("lat"))
    longitude = event.get("longitude", event.get("lon"))
    if latitude is None or longitude is None:
        continue

    popup_html = f"""
    <b>{event.get('company', '(unknown company)')}</b><br>
    Ticker: {event.get('ticker') or '(unknown)'}<br>
    Industry: {event.get('industry', '(unknown)')}<br>
    Filing date: {event.get('file_date', '(unknown)')}<br>
    Event type: {event.get('event_type', '(unknown)')}<br>
    Summary: {event.get('summary', '')}<br>
    <a href="{event.get('url', '#')}" target="_blank">SEC filing</a>
    """

    folium.Marker(
        location=[latitude, longitude],
        popup=folium.Popup(popup_html, max_width=350),
        icon=folium.Icon(
            color=INDUSTRY_COLORS.get(event.get("industry"), "gray"),
            icon=EVENT_ICONS.get(event.get("event_type"), "info-circle"),
            prefix="fa",
        ),
    ).add_to(m)
    marker_count += 1

legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
            background: white; border: 1px solid #999; border-radius: 6px;
            padding: 10px 12px; font-size: 13px; line-height: 1.45;
            box-shadow: 0 1px 4px rgba(0,0,0,0.25);">
  <b>Industry</b><br>
  <span style="color:#00008b;">&bull;</span> Financial Services<br>
  <span style="color:#008000;">&bull;</span> Travel and Hospitality<br>
  <br>
  <b>Event type</b><br>
  + Opening<br>
  &times; Closing<br>
  &harr; Relocation<br>
  &#10530; Expansion<br>
  i Other
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

print(f"Mapped {marker_count} events")
display(m)


Mapped 24 events


### Export the map to `maps/mp03_map_team_16.html`

In [19]:
OUTPUT_PATH = "../maps/mp03_map_team_16.html" if os.path.basename(os.getcwd()) == "notebooks" else "maps/mp03_map_team_16.html"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
m.save(OUTPUT_PATH)
print(f"Map saved to {OUTPUT_PATH}")


Map saved to ../maps/mp03_map_team_16.html


## MP03 Methodology - Team 16

### Financial Services Pipeline - Chinmoy Chowdhury

#### 6.1 Ticker-list rationale

The Financial Services ticker list was extended from 14 to 28 companies. The original seed focused on large national banks, payments, insurance, and asset management firms, but the first live runs showed that many physical branch announcements come from regional and community banks. The team kept the original seed, added Goldman Sachs (GS), Morgan Stanley (MS), and Charles Schwab (SCHW) for capital-markets and brokerage coverage, and added NKSH, UNTY, FMCB, HNVR, MCB, FUNC, WSBK, WSBC, PGC, JUVF, and BRO because those companies produced finance-sector branch, office, or financial-center filings during tuning.

#### 6.2 Search-phrase rationale

Two Financial Services phrases were added: "financial center" and "wealth management office". These phrases catch banking and brokerage location announcements that do not always use the word branch. The original data-center phrase produced many false positives before ticker filtering, so the final pipeline strictly filters every candidate by ticker before sending it to Claude.

#### 6.3 Financial Services window results

| industry | window_days | candidate_count | event_count | estimated_cost_usd |
|---|---:|---:|---:|---:|
| Financial Services | 30 | 12 | 11 | 0.0320 |

The 30-day Financial Services trial passed the 8-event target and stayed well under the cost ceiling. For the final common 180-day window, Financial Services produced 25 ticker-filtered candidates, 18 classified events, and 15 geocoded map events.

#### 6.4 Financial Services classification quality

In the 30-day window trial, 12 ticker-filtered candidates produced 11 classified location events, and 10 geocoded successfully. In the final 180-day common-window run, 25 candidates produced 18 classified events and 15 geocoded events. The stricter ticker filter removed thousands of unrelated EDGAR search hits before classification, which improved sector accuracy and prevented unrelated companies from appearing as Financial Services markers.

### Travel and Hospitality Pipeline - Rahim

#### 6.1 Ticker-list rationale

The Travel and Hospitality ticker list was extended from 14 to 21 companies. The original seed covered hotels, cruise lines, airlines, and online travel companies, but live tuning showed that many recent location events came from gaming, entertainment venues, hospitality real estate, and lodging-adjacent operators. The team added PENN, BALY, GLPI, VENU, CWD, HHH, and TH to capture casino openings, venue openings, hotel developments, and hospitality property activity.

#### 6.2 Search-phrase rationale

The seeded phrases were kept because they cover the main location events for this industry: new hotels, resort openings, property openings, brand conversions, new routes, gateways, terminals, and grand openings. The final code pairs those broad phrases with strict ticker filtering so unrelated companies found by generic phrases such as "grand opening" are removed before classification.

#### 6.3 Travel and Hospitality window results

| industry | window_days | candidate_count | event_count | estimated_cost_usd |
|---|---:|---:|---:|---:|
| Travel and Hospitality | 30 | 2 | 2 | 0.0052 |
| Travel and Hospitality | 60 | 3 | 3 | 0.0077 |
| Travel and Hospitality | 90 | 7 | 7 | 0.0187 |
| Travel and Hospitality | 180 | 10 | 9 | 0.0264 |

The 30-day, 60-day, and 90-day windows did not reach the 8-event target. The 180-day window reached the target with 9 classified location events, all of which geocoded successfully. The Travel and Hospitality cumulative window-tuning cost was about $0.0580. Including the Financial Services 30-day trial, the cumulative window-tuning cost across both industries was about $0.0899, below the $3.00 limit.

#### 6.4 Travel and Hospitality classification quality

The final 180-day Travel and Hospitality run produced 10 ticker-filtered candidates, 9 classified events, and 9 geocoded events. The stricter ticker filter removed thousands of nonmatching search hits before Claude classification, which fixed the earlier problem where unrelated companies appeared in the Travel and Hospitality layer.

#### 6.5 Limitations

The pipeline is limited to US geocoding, so international hotel, airline, cruise, and banking activity can be dropped even when the filing is relevant. The two sectors also use different location language: Financial Services uses branch, office, and financial-center wording, while Travel and Hospitality uses hotel, venue, property, route, and grand-opening language. Another limitation is that the exhibit text is truncated to 8,000 characters, so long filings may lose location details that appear later in the document. The final map contains 24 geocoded events: 15 Financial Services events and 9 Travel and Hospitality events.


---

## 7. Comparative Reflection

The final map suggests that Financial Services and Travel and Hospitality deploy physical capacity in different ways. Financial Services events are mostly branch, office, and financial-center announcements from regional and community banks. These locations are tied to local customer access, deposit gathering, commercial relationships, and market coverage. Even though many financial services have moved online, banks still use physical locations to build trust and serve specific local markets.

Travel and Hospitality events look more tied to destination demand and customer experiences. The final Travel and Hospitality layer includes casino, hotel, entertainment venue, and hospitality-property activity. These projects usually add capacity where companies expect people to physically gather, stay, visit, or spend leisure dollars. That makes the geography feel more dependent on tourism, entertainment districts, regional growth markets, and large physical venues.

The window experiments also show a detection difference. Financial Services reached the 8-event target in a 30-day window, while Travel and Hospitality needed a 180-day window. This does not prove that finance has more location activity overall. It mainly shows that the Financial Services phrase and ticker list found relevant branch filings more quickly, while Travel and Hospitality announcements were less frequent among the selected public companies.

There are important limitations. The pipeline only geocodes US locations, so international hotels, routes, resorts, offices, and branches may be missing. The strict ticker filter improved accuracy, but it also means the results depend heavily on the team’s selected ticker lists. Some valid industry events from companies outside the list are intentionally excluded. Even with those limits, the comparison supports the main interpretation: Financial Services geography is about local market coverage and branch economics, while Travel and Hospitality geography is about adding customer-facing capacity where travel and entertainment demand exists.


---

## 8. Pre-Submission Verification

Before the integrator submits, confirm each of the following:

- [ ] Notebook restarts cleanly and runs end-to-end in Colab. Local `nbconvert` execution completed successfully; run one final Colab restart/run before submission.
- [x] No committed API keys, no hard-coded credentials, no leftover debug prints.
- [x] `window_results` table is populated with at least one row per (industry, window) trial actually run.
- [x] Both industries reach at least 8 location events at the chosen window, OR a 360-day trial was run for both and the short-fall is acknowledged in Section 6.
- [x] Cumulative window-tuning cost is at or below $3.00.
- [x] Integrated map renders inline AND is exported to `maps/mp03_map_team_16.html`.
- [x] Every marker has a popup with all required fields and a working SEC hyperlink.
- [x] Industry is visually distinguishable from event type on the map.
- [x] Methodology appears both in this notebook and at `methodology/mp03_methodology_team_16.md`.
- [x] Comparative reflection appears both in this notebook and at `reflections/mp03_reflection_team_16.md`.
- [ ] Team branch name is exactly `mp/03-industry-comparison-team-16` and submission tag `mp03-team-16` is pushed. Branch name is correct; tag still needs to be created and pushed.
- [ ] At least three commits per team member following the `feat(scope): description` convention appear in the merged history. Current visible history does not show three Rahim-authored `feat(...)` commits.
- [ ] Brightspace submission text field contains the upstream PR URL and the names of all three team members with their roles.
